In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torch
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

With MDTA the attention product grows linearly in $N=H*W$ instead of $N^2$
We have here (N,C) @ (C,C) instead of (C,N) @ (N,N) and thus it grows in $O(C^2 N)$ instead of $O(C N^2)$

In [2]:
x_t = torch.randn(5, 4, 32, 64).transpose(-2,-1)
conv_t = nn.Conv2d(5,5, 3)
conv_tg = nn.Conv2d(5,5,3, groups=5)
print(f" weights dense conv:{conv_t.weight.shape}")
print(f" weights groups conv:{conv_tg.weight.shape}")
print((x_t).shape)

 weights dense conv:torch.Size([5, 5, 3, 3])
 weights groups conv:torch.Size([5, 1, 3, 3])
torch.Size([5, 4, 64, 32])


In [10]:
class MDTA(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.alpha = nn.Parameter(torch.ones(num_heads, 1, 1))
        assert dim%num_heads==0, "C has to be divisible by num_heads"
        self.qkv_p = nn.Conv2d(dim, 3*dim, kernel_size=1, bias=False)
        self.qkv_d = nn.Conv2d(3*dim, 3*dim, kernel_size=1, groups=3*dim, bias=False)
         
        self.proj = nn.Conv2d(dim, dim, kernel_size=1)
    
    def forward(self, x):
        """
        B = batch_size
        inputs:
        query = key = value = (B, C, H, W)
        x = entry to get the skip

        """
        B, C, H, W = x.shape
        N = H*W
        # efficient implementation of the convolutions
        q, k, v = self.qkv_d(self.qkv_p(x)).chunk(3, dim=1)

        k = k.reshape(B, self.num_heads, C//self.num_heads, N) # (*, H, C//H, H*W)
        q = q.reshape(B, self.num_heads, C//self.num_heads, N) # (*, H, C//H, H*W)
        v = v.reshape(B, self.num_heads, C//self.num_heads, N) # (*, H, C//H, H*W)
        

        
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        #multiplying by alpha is more stable and equivalent 
        dot = F.softmax((q @ k.transpose(-2,-1))*self.alpha, -1) # (dim, dim)
        out = dot @ v 
        out = out.reshape(B, C, H, W)
        return x + self.proj(out)

x_t = torch.randn(5, 128, 64, 64)
mdta = MDTA(128, 8)
x_t = mdta(x_t)
x_t.shape

torch.Size([5, 128, 64, 64])

16

TESTER sans normalize ? Voir l'effet de normalize?